# MIMIC-IV Continuous IQN Training Pipeline

Full end-to-end training pipeline for the continuous Dead-End Discovery (DeD) agent on MIMIC-IV sepsis/hypotension data.

**Pipeline steps:**
1. Clone repo + install dependencies (CSV data is included in the repo)
2. Build raw NPZ files from CSVs
3. Preprocess: masks, intensities, normalization, rectilinear interpolation
4. Train NCDE state encoder (fixed hyperparameters, no Ax search)
5. Encode all trajectories into NCDE hidden states
6. Train continuous IQN (D-network + R-network)
7. Download trained models

**Requirements:** Add your GitHub token as a Colab secret named `GITHUB_TOKEN`.

## Step 1: Clone repository and install dependencies

In [ ]:
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
!git clone -b mimic https://{token}@github.com/jaym-01/ContinuousDeD.git tmp
%pip install -r tmp/requirements.txt -q
%pip install torchcde ax-platform pyyaml -q

In [ ]:
# Set working directory to the cloned repo
%cd /content/tmp

import os, sys
sys.path.insert(0, '/content/tmp')
print("Working directory:", os.getcwd())

## Step 2: Build raw NPZ files from CSVs

The trajectory CSVs are included in the repo. This converts them into the NPZ format expected by the preprocessing pipeline.

**Output:**
- `data/continuous_mimic/reduced_format.npz` — main cohort
- `data/continuous_mimic/reduced_format_overlapCohort.npz` — overlap cohort

In [ ]:
import pandas as pd
import numpy as np
import gzip

# Column definitions matching the MIMIC notebook
STATIC_COLS   = ['o:gender', 'o:admission_age', 'o:height', 'o:weight']
TEMPORAL_COLS = [
    'm:timestep',
    'o:heart_rate', 'o:sbp', 'o:dbp', 'o:mbp', 'o:resp_rate', 'o:temperature',
    'o:glucose', 'o:so2', 'o:po2', 'o:pco2', 'o:fio2', 'o:pao2fio2ratio',
    'o:ph', 'o:baseexcess', 'o:chloride', 'o:calcium', 'o:potassium', 'o:sodium',
    'o:lactate', 'o:hematocrit', 'o:hemoglobin', 'o:platelet', 'o:wbc', 'o:albumin',
    'o:aniongap', 'o:bicarbonate', 'o:pt', 'o:ptt', 'o:gcs',
    'o:spo2', 'o:bun', 'o:creatinine', 'o:inr', 'o:bilirubin', 'o:alt', 'o:ast',
    'o:UO', 'o:ventilation'
]
ACTION_COLS   = ['a:action_fluid', 'a:action_vaso']
OUTCOME_COL   = ['r:reward']

def read_csv_auto(path):
    """Read CSV whether gzip-compressed or plain text."""
    try:
        with gzip.open(path, 'rt') as f:
            return pd.read_csv(f)
    except Exception:
        return pd.read_csv(path)

def csv_to_npz(csv_path, npz_path):
    """Convert a trajectory CSV to the raw NPZ format required by preprocess_ncde_data.py."""
    print(f"Reading {csv_path} ...")
    df = read_csv_auto(csv_path)
    print(f"  {len(df)} rows, {df['m:stay_id'].nunique()} unique stays")

    # One row per stay (first timestep) for static features
    static_frame = (
        df[['m:stay_id'] + STATIC_COLS]
        .groupby('m:stay_id', sort=True)
        .first()
    )

    unique_ids = static_frame.index.tolist()

    static_data  = []
    temporal_data = []
    action_data  = []
    outcome_data = []

    df_by_stay = df.set_index('m:stay_id')

    for sid in unique_ids:
        rows = df_by_stay.loc[[sid]].sort_values('m:timestep')

        static_data.append(static_frame.loc[sid].values.astype(np.float32))
        temporal_data.append(rows[TEMPORAL_COLS].values.astype(np.float32))
        action_data.append(rows[ACTION_COLS].values.astype(np.float32))
        outcome_data.append(rows[OUTCOME_COL].values.astype(np.float32))

    print(f"  Saving to {npz_path} ...")
    np.savez(
        npz_path,
        static_data   = np.stack(static_data),                    # (N, 4)
        temporal_data = np.array(temporal_data, dtype=object),    # object (N,) → each (T, 39)
        action_data   = np.array(action_data,   dtype=object),    # object (N,) → each (T, 2)
        outcome_data  = np.array(outcome_data,  dtype=object),    # object (N,) → each (T, 1)
        static_columns  = STATIC_COLS,
        temporal_columns = TEMPORAL_COLS,
        stay_id         = np.array(unique_ids, dtype=np.float64),
    )
    print(f"  Done ({len(unique_ids)} patients).\n")


DATA_DIR = 'data/continuous_mimic'

csv_to_npz(
    os.path.join(DATA_DIR, 'final_trajs_noFill.csv'),
    os.path.join(DATA_DIR, 'reduced_format.npz')
)

csv_to_npz(
    os.path.join(DATA_DIR, 'final_trajs_overlapCohort_noFill.csv'),
    os.path.join(DATA_DIR, 'reduced_format_overlapCohort.npz')
)

print("Both NPZ files created successfully.")

## Step 3: Preprocess — masks, intensities, normalization, rectilinear interpolation

Runs `preprocess_ncde_data.py` which:
- Computes observation masks and measurement intensities
- Forward-fills NaN values
- Z-score normalizes using training-set statistics
- Creates stratified 70/15/15 train/val/test splits
- Runs rectilinear interpolation (doubles timesteps: T → 2T-1)

**Output:** `data/continuous_mimic/rectilinear_processed/improved-neural-cdes_data.npz`

In [ ]:
!python preprocess_ncde_data.py

## Step 4: Train NCDE state encoder

Trains the Neural Controlled Differential Equation (NCDE) to encode variable-length patient
time-series into fixed-size hidden state vectors.

**Architecture (fixed, well-tuned hyperparameters):**
- `hidden_dim = 64` — NCDE hidden state dimension
- `hidden_hidden_dim = 64` — NCDE vector field width
- `pred_num_layers = 3` — prediction head depth
- `pred_num_units = 128` — prediction head width
- 50 epochs on combined train+val set

**Output:** `data/continuous_mimic/ncde_output/best_model.pt`

In [ ]:
import torch
import yaml
import random

from ncde_utils import load_data, evaluator
from ncde import NeuralCDE

# ── Configuration ─────────────────────────────────────────────────────────────
NCDE_PARAMS = {
    'hidden_dim':        64,
    'hidden_hidden_dim': 64,
    'pred_num_layers':   3,
    'pred_num_units':    128,
    'lr':                5e-4,
    'num_training_epochs': 50,
    'lr_step_size':      20,
    'lr_gamma':          0.5,
}
CHECKPOINT_EVERY = 5  # save NCDE checkpoint every N epochs

config_dict = yaml.safe_load(open('./configs/ncde_config.yaml', 'r'))
DATA_DIR    = config_dict['data_dir']
OUTPUT_DIR  = config_dict['output_dir']
BATCH_SIZE  = config_dict['batch_size']
SEED        = config_dict.get('seed', 2022)

os.makedirs(OUTPUT_DIR, exist_ok=True)
NCDE_CKPT_PATH   = os.path.join(OUTPUT_DIR, 'ncde_checkpoint.pt')
NCDE_BEST_PATH   = os.path.join(OUTPUT_DIR, 'best_model.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

torch.manual_seed(SEED)
random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

# ── Load data ──────────────────────────────────────────────────────────────────
(train_loader, val_loader, _), input_dim, action_dim, static_dim, output_dim = load_data(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE
)
(combined_loader, _, _), _, _, _, _ = load_data(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, combine_train_val=True, shuffle=True
)
print(f"Input dim: {input_dim}, Action dim: {action_dim}, Static dim: {static_dim}, Output dim: {output_dim}")

# ── Initialise model, optimizer, scheduler ────────────────────────────────────
model = NeuralCDE(
    input_dim, NCDE_PARAMS['hidden_dim'], output_dim, static_dim, action_dim,
    hidden_hidden_dim=NCDE_PARAMS['hidden_hidden_dim'],
    pred_num_layers=NCDE_PARAMS['pred_num_layers'],
    pred_num_units=NCDE_PARAMS['pred_num_units'],
    return_sequences=True, device=device
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=NCDE_PARAMS['lr'], amsgrad=True)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=int(NCDE_PARAMS['lr_step_size']),
    gamma=NCDE_PARAMS['lr_gamma']
)

# ── Resume from checkpoint if one exists ──────────────────────────────────────
start_epoch = 0
if os.path.exists(NCDE_CKPT_PATH):
    print(f"Resuming from checkpoint: {NCDE_CKPT_PATH}")
    ckpt = torch.load(NCDE_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    print(f"  Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found — training from scratch.")

# ── Training loop with periodic checkpointing ─────────────────────────────────
num_epochs = NCDE_PARAMS['num_training_epochs']
model.train()

for epoch in range(start_epoch, num_epochs):
    loss_pred = 0
    print(f"Epoch {epoch+1}/{num_epochs}", end=" ... ")

    for inputs, masks, lengths, targets, _, _ in combined_loader:
        static, temporal, actions = inputs
        static  = static.to(device)
        temporal = temporal.to(device)
        actions = actions.to(device)
        masks   = masks.to(device)
        lengths = lengths.to(device)
        targets = targets.to(device)

        max_length = int(lengths.max().item())
        temporal = temporal[:, :(2*max_length)-1, :]
        actions  = actions[:, :max_length, :]
        targets  = targets[:, :max_length, :]
        masks    = masks[:, :max_length, :-1]

        preds, _ = model((static, temporal, actions))
        loss = model.calculate_loss(preds, targets, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_pred += loss.detach().cpu().item()

    print(f"loss = {loss_pred:.4f}")

    # Periodic checkpoint (includes optimizer + scheduler state for resuming)
    if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == num_epochs:
        torch.save({
            'epoch':      epoch,
            'model':      model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'scheduler':  scheduler.state_dict(),
            'hyperparameters': NCDE_PARAMS,
        }, NCDE_CKPT_PATH)
        print(f"  Checkpoint saved (epoch {epoch+1})")

# ── Save final model in the format expected by encode_data.py ─────────────────
torch.save({'model': model.state_dict(), 'hyperparameters': NCDE_PARAMS}, NCDE_BEST_PATH)
print(f"\nNCDE model saved to: {NCDE_BEST_PATH}")

## Step 5: Encode all trajectories into NCDE hidden states

Runs the trained NCDE over every patient trajectory to produce fixed-size hidden state sequences.
These become the state representations for the offline RL training.

**Output:** `data/continuous_mimic/rectilinear_processed/encoded_{train,validation,test,overlap}.npz`

In [ ]:
!python encode_data.py

In [ ]:
# Quick sanity check on the encoded data
import numpy as np

DATA_DIR = 'data/continuous_mimic/rectilinear_processed'
for split in ['train', 'validation', 'test']:
    npz = np.load(os.path.join(DATA_DIR, f'encoded_{split}.npz'), allow_pickle=True)
    s = npz['states']
    a = npz['actions']
    r = npz['rewards']
    print(f"  {split:12s}: states {s.shape}, actions {a.shape}, rewards {r.shape}")
    print(f"               actions dtype={a.dtype}, range=[{a.min():.3f}, {a.max():.3f}]")

## Step 6: Train continuous IQN (D-network + R-network)

Trains the offline continuous IQN agent:
- **D-network** (`sided_Q=negative`): learns to identify dead-end states
- **R-network** (`sided_Q=positive`): learns to identify recovery states

**Optimized hyperparameters:**
- Wider/deeper network: 256 hidden units × 3 layers
- 64 quantile samples during training, 128 random-shooting action candidates
- CQL regularization (weight=0.5) to prevent OOD Q-value extrapolation
- 200 epochs, batch size 256, lr=1e-4
- Oversampling: 2 positive (survivor) + 4 negative (death) terminals per minibatch

**Output:** `runs/iqn_continuous_mimic/best_q_parameters{negative,positive}.pt`

In [ ]:
# Quick smoke test first — verifies the pipeline runs without errors (~30 seconds)
!python train_rl.py -c iqn_continuous_mimic -o smoke_test True

In [ ]:
# Full training with optimized hyperparameters
# Overrides are validated against the YAML config before training starts
!python train_rl.py -c iqn_continuous_mimic \
    -o num_q_hidden_units 256 \
    -o num_q_layers 3 \
    -o num_iqn_samples_train 64 \
    -o num_iqn_samples_est 64 \
    -o K_actions 128 \
    -o use_cql True \
    -o cql_weight 0.5 \
    -o train_batch_size 256 \
    -o num_epochs 200 \
    -o lr 0.0001 \
    -o num_ps 2 \
    -o num_ns 4

In [ ]:
# Verify checkpoint files exist
import glob

ckpt_dir = 'runs/iqn_continuous_mimic'
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, '*.pt')))
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f"  {c}  ({size_mb:.1f} MB)")

if not ckpts:
    print("No checkpoints found — check training logs above.")

## Step 7: Download checkpoints

Run this cell at any time — before, during, or after training — to download everything needed to resume on a new session.

| Archive | Contents | Used for |
|---------|----------|----------|
| `iqn_continuous_mimic_run.zip` | All IQN `.pt` files (saved every epoch) | Resume IQN or run inference |
| `ncde_model.zip` | `ncde_checkpoint.pt` (resumable) + `best_model.pt` | Resume NCDE training or encode |
| `encoded_data.zip` | `encoded_{train,validation,test,overlap}.npz` | Skip re-encoding on resume |

**To resume after a disconnection:**
1. Run Steps 1–2 (clone + build NPZs) — fast, always needed
2. Upload your downloaded zips and extract to the right paths:
   ```python
   !unzip ncde_model.zip -d /content/tmp/data/continuous_mimic/ncde_output/
   !unzip encoded_data.zip -d /content/tmp/data/continuous_mimic/rectilinear_processed/
   !unzip iqn_continuous_mimic_run.zip -d /content/tmp/runs/iqn_continuous_mimic/
   ```
3. Re-run whichever step was interrupted — NCDE and IQN training both detect and resume from their checkpoints automatically

In [ ]:
import shutil, glob
from google.colab import files

def download_zip(folder_path, archive_name):
    """Zip a folder and download it."""
    zip_path = f"/content/{archive_name}"
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', root_dir=folder_path)
    files.download(zip_path)
    print(f"Downloaded: {archive_name}")

# ── IQN checkpoints (D-network + R-network, saved every epoch) ────────────────
download_zip('/content/tmp/runs/iqn_continuous_mimic', 'iqn_continuous_mimic_run.zip')

# ── NCDE checkpoints (resumable checkpoint + final best_model) ────────────────
download_zip('/content/tmp/data/continuous_mimic/ncde_output', 'ncde_model.zip')

# ── Encoded data (skip re-encoding on resume if NCDE is already done) ─────────
download_zip('/content/tmp/data/continuous_mimic/rectilinear_processed', 'encoded_data.zip')

## (Optional) Plot training and validation loss curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ckpt_dir = 'runs/iqn_continuous_mimic'
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sided_Q in zip(axes, ['negative', 'positive']):
    loss_path = os.path.join(ckpt_dir, f'q_losses_{sided_Q}.npy')
    ckpt_path = os.path.join(ckpt_dir, f'q_parameters{sided_Q}.pt')

    if os.path.exists(loss_path):
        train_loss = np.load(loss_path)
        ax.plot(train_loss, label='Train loss')

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_loss = ckpt.get('validation_loss', [])
        if val_loss:
            ax.plot(val_loss, label='Val loss', linestyle='--')

    network_name = 'D-network (dead-end)' if sided_Q == 'negative' else 'R-network (recovery)'
    ax.set_title(network_name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Continuous IQN Training Curves — MIMIC-IV Sepsis', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")